# Pipeline clustering perangkat


## EDA

### Input citra


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from PIL import Image
from sklearn.pipeline import Pipeline
from joblib import dump

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from bdc.clustering import FusedKMeans
from bdc.config import ACTIVE_MODEL, BACKBONES, K
from bdc.embeddings import FrozenEmbeddingExtractor
from bdc.preprocessing import IMAGE_SIZE, JPEG_QUALITY, harmonize

data = ""


In [ ]:
if not data.strip():
    raise ValueError("Isi variabel data dengan path folder atau file citra.")

data_path = Path(data).expanduser()
if not data_path.is_absolute():
    data_path = (ROOT / data_path).resolve()

image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
if data_path.is_file():
    image_paths = [data_path] if data_path.suffix.lower() in image_extensions else []
else:
    image_paths = sorted(
        path for path in data_path.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )

if not image_paths:
    raise FileNotFoundError(f"Tidak ada citra yang ditemukan di {data_path}")

formats = pd.Series([path.suffix.lower() for path in image_paths])
display(formats.value_counts().rename_axis("format").to_frame("jumlah citra"))
dimensions = []
for path in image_paths:
    with Image.open(path) as image:
        dimensions.append((image.width, image.height))
dimensions = pd.DataFrame(dimensions, columns=["width", "height"])
display(dimensions.describe().round(0))
for path in image_paths[:4]:
    with Image.open(path) as image:
        display(image.convert("RGB"))


## Preprocessing

### Resize dan JPEG


In [ ]:
with Image.open(image_paths[0]) as image:
    original = image.convert("RGB")
    processed = harmonize(original)

display(Markdown("**Input**"))
display(original)
display(Markdown(f"**Setelah preprocessing** · {IMAGE_SIZE} × {IMAGE_SIZE} px · JPEG q{JPEG_QUALITY}"))
display(processed)


## Feature extraction

### Frozen backbone


In [ ]:
models = pd.DataFrame([
    {"backbone": name, "model": item["model"], "input_size": item["size"]}
    for name, item in BACKBONES.items()
])
display(models)

pipeline = Pipeline([
    ("feature_extraction", FrozenEmbeddingExtractor(preset=ACTIVE_MODEL, batch_size=16)),
    ("clustering", FusedKMeans(k=K, preset=ACTIVE_MODEL)),
])
display(pipeline)


## Clustering

### Fit K-Means


In [ ]:
pipeline.fit(image_paths)
clusterer = pipeline.named_steps["clustering"]
labels = clusterer.labels_

counts = pd.Series(labels).value_counts().sort_index()
counts.index = [f"C{cluster:02d}" for cluster in counts.index]
display(counts.rename("jumlah citra").to_frame())


## Hasil

### Contoh citra per cluster


In [ ]:
features = pipeline.named_steps["feature_extraction"].features_
space = clusterer.transform(features)
distances = clusterer.model.transform(space)

for cluster in range(K):
    members = np.flatnonzero(labels == cluster)
    if members.size:
        index = members[np.argmin(distances[members, cluster])]
        display(Markdown(f"**C{cluster:02d}** · {image_paths[index].name}"))
        with Image.open(image_paths[index]) as image:
            display(image.convert("RGB"))


### Simpan hasil


In [ ]:
output = ROOT / "run_outputs"
output.mkdir(parents=True, exist_ok=True)

relative_paths = [
    str(path.relative_to(data_path)) if data_path.is_dir() else path.name
    for path in image_paths
]
assignments = pd.DataFrame({"image": relative_paths, "cluster": labels})
assignments.to_csv(output / "cluster_assignments.csv", index=False)
dump(clusterer, output / "cluster_model.joblib")
display(assignments.head())
print(f"Hasil disimpan di {output}")
